In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../')))

from lib.preprocessor import Preprocessor
from lib.data_preparation import DataPreparation
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

import pandas as pd
pd.set_option('display.max_colwidth', 100)

df = DataPreparation.load()
df_train, df_test = DataPreparation.save_train_test_split(df, save=False)


X = df_train['tweet'].values.tolist()
y = df_train['sentiment'].values.tolist()


X_tranformed, _ = Preprocessor.process(
        raws=X,
        processing_params={
            'method':'lemmatize',
            'vectorization': 'word2vec'
        },
        show_trans=False,
        stopwords=True
    )

model = OneVsRestClassifier(LogisticRegression(max_iter=200, penalty='l2', solver='liblinear'))
model.fit(X_tranformed, y)

X_test = df_test['tweet'].values.tolist()
y_test = df_test['sentiment'].values.tolist()

X_test_transformed, _ = Preprocessor.process(
        raws=X_test,
        processing_params={
            'method':'lemmatize',
            'vectorization': 'word2vec'
        },
        show_trans=False,
        stopwords=True
)
y_pred = model.predict(X_test_transformed)
y_prob = model.predict_proba(X_test_transformed)

acc = accuracy_score(y_pred=y_pred, y_true=y_test)
auc = roc_auc_score(y_score=y_prob, y_true=y_test, multi_class='ovr')

result = pd.DataFrame({
    'model':['Logistic regression'],
    'preprocessing_method':['lemmatize + Word2vec'],
    'accuracy':[acc],
    'AUC':[auc],
    'best_hyper_params':["{'estimator__penalty': 'l2', 'estimator__solver': 'liblinear'}"]
})
print(result)

